# ML-09 — Validation and Research Claim Audit

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Ubaidrees/flyrank-ml-tasks/blob/main/work/notebooks/w06_validation_audit.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Two paper findings + my methodology questions

*Pick two findings from the FlyRank research paper. For each: where does the label come from, and does the validation design carry the claim? Constructive tone.*

Finding #2 — "The Content Performance Curve" (health score by content age, showing a 365+ day "rebound" to 25.1)

Methodology question: Is this rebound measured longitudinally (the same pages tracked as they age past 365 days) or cross-sectionally (different pages, currently at different ages, compared to each other)? If cross-sectional, the 365+ bucket only contains pages that survived long enough to still be active and happened to get refreshed — meaning the apparent "rebound" could reflect which old pages got editorial attention, not proof that age itself reverses decline. Notably, the paper already applies exactly this caution to an adjacent cell in Finding #8 ("a very small active-content survivor sample"), so it would strengthen Finding #2 to state the same caveat explicitly rather than only nearby.

Finding #4 — "The Freshness Multiplier" (365+ day pages that were refreshed show a 3.2x health boost and 57x more impressions)

Methodology question: Where does the "refreshed" label come from — a random sample of stale pages, or a set editors specifically chose to refresh (likely because they already judged the page worth the effort — a high-value keyword, prior authority, business priority)? If refreshed pages were selected rather than randomly assigned, part of the 3.2x/57x boost may reflect the selection criteria itself (editors picking promising candidates), not the refresh action. A stronger design would compare refreshed pages against a matched set of similarly-aged, similarly-authoritative pages that were not refreshed — not just against the general declining population.

In [5]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 2. My model under an honest split (before/after)

A naive random split showed 31 of 31 test clients also present in training — complete client overlap, not partial. This inflated AUC from 0.604 (honest, client-grouped) to 0.745 — a 0.141-point overstatement. The naive number would have looked like a genuinely strong model; the honest number reveals it's a real but more modest signal, since the model was never actually tested on a client it hadn't seen.

In [6]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split, GroupShuffleSplit
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import roc_auc_score

# ---- Rebuild the dataset and features (same as ML-08) ----
df = pd.read_csv("https://raw.githubusercontent.com/Ubaidrees/flyrank-ml-tasks/main/data/raw/content_refresh_anonymized.csv")
df['is_declining_label'] = (df['trend_direction'] == 'down').astype(int)
df = df[(df['impressions_90d'] > 0) & (df['content_age_days'] >= 90)].copy()
df = df.drop_duplicates(subset='content_id')

feature_cols = [c for c in [
    'impressions_90d', 'clicks_90d', 'sessions_90d', 'ai_sessions_90d',
    'content_age_days', 'days_since_last_update',
    'ctr', 'avg_position', 'engagement_rate', 'scroll_rate',
    'word_count', 'ai_traffic_pct'
] if c in df.columns]

X = df[feature_cols].fillna(0)
y = df['is_declining_label']
groups = df['client_id']

# ---- AFTER: honest, client-grouped split (same design as ML-08) ----
gss = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=42)
train_idx, test_idx = next(gss.split(X, y, groups))
X_train, X_test = X.iloc[train_idx], X.iloc[test_idx]
y_train, y_test = y.iloc[train_idx], y.iloc[test_idx]

model = RandomForestClassifier(n_estimators=300, max_depth=8, random_state=42, class_weight='balanced')
model.fit(X_train, y_train)
model_auc = roc_auc_score(y_test, model.predict_proba(X_test)[:, 1])

grouped_overlap = set(df.iloc[train_idx]['client_id']) & set(df.iloc[test_idx]['client_id'])

# ---- BEFORE: naive random split — no grouping ----
X_train_naive, X_test_naive, y_train_naive, y_test_naive, idx_train_naive, idx_test_naive = train_test_split(
    X, y, df.index, test_size=0.2, random_state=42, stratify=y
)
model_naive = RandomForestClassifier(n_estimators=300, max_depth=8, random_state=42, class_weight='balanced')
model_naive.fit(X_train_naive, y_train_naive)
naive_auc = roc_auc_score(y_test_naive, model_naive.predict_proba(X_test_naive)[:, 1])

naive_test_clients = set(df.loc[idx_test_naive, 'client_id'])
naive_train_clients = set(df.loc[idx_train_naive, 'client_id'])
naive_overlap = naive_test_clients & naive_train_clients

print(f"BEFORE (naive random split): AUC = {naive_auc:.3f}")
print(f"Client overlap in naive split: {len(naive_overlap)} of {len(naive_test_clients)} test clients also in training")
print(f"\nAFTER (client-grouped split): AUC = {model_auc:.3f}")
print(f"Client overlap in grouped split: {len(grouped_overlap)} (should be 0)")
print(f"\nInflation from the naive split: {naive_auc - model_auc:+.3f}")

BEFORE (naive random split): AUC = 0.745
Client overlap in naive split: 31 of 31 test clients also in training

AFTER (client-grouped split): AUC = 0.604
Client overlap in grouped split: 0 (should be 0)

Inflation from the naive split: +0.141


## 3. Leakage audit
trend_direction (the label) is defined by FlyRank's own methodology as "30d-vs-prev-30d impression change" — and impressions_90d structurally includes that same final 30-day window. Removing it dropped AUC from 0.604 to 0.551 (a 0.053 drop), confirming it does carry some leaked signal, not pure independent prediction. However, AUC 0.551 remains above the 0.5 random baseline, meaning the model retains real, non-leaked signal from the other 11 features even without it. Verdict: impressions_90d is partially, not fully, a leakage artifact — worth flagging in any reported feature-importance ranking rather than removing outright.

In [7]:
no_impressions_features = [c for c in feature_cols if c != 'impressions_90d']
X_train_no_imp = X_train[no_impressions_features]
X_test_no_imp = X_test[no_impressions_features]

model_no_imp = RandomForestClassifier(n_estimators=300, max_depth=8, random_state=42, class_weight='balanced')
model_no_imp.fit(X_train_no_imp, y_train)
auc_no_imp = roc_auc_score(y_test, model_no_imp.predict_proba(X_test_no_imp)[:, 1])

print(f"Model WITH impressions_90d: AUC = 0.604")
print(f"Model WITHOUT impressions_90d: AUC = {auc_no_imp:.3f}")
print(f"Drop from removing it: {0.604 - auc_no_imp:+.3f}")

Model WITH impressions_90d: AUC = 0.604
Model WITHOUT impressions_90d: AUC = 0.551
Drop from removing it: +0.053


## 4. Claim rewrite
Before (overclaimed): "Our model predicts content decline with 74.5% AUC accuracy."

After (safe language): "Under a client-grouped validation split — a stricter test that holds out entire clients rather than individual rows — the model measured an observed AUC of 0.604, a directional signal useful for decision-support triage. A naive split that allowed client overlap inflated this to 0.745, which would have overstated real-world performance by 14 points. The honest number, not the naive one, is what should inform any prioritization decision."

In [8]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.